# De las flores a las decisiones: vamos a entender ADL

Cuatro medidas, tres especies y dos ejes: Iris nos permite ver cómo una combinación de variables separa grupos. Seguiremos cada operación hasta una predicción y comprobaremos qué parte de la historia cuenta una figura. Tiempo orientativo: 75–90 minutos, repartidos en varios bloques.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlacasa/EstadisticaQ2/blob/main/AnalisisDiscriminanteLineal.ipynb)

[Itinerario de prácticas](https://github.com/mlacasa/EstadisticaQ2/blob/main/README.md) · Tema 2

## Antes de empezar

1. Guarda tu copia en Drive desde el menú **Archivo** de Colab.
2. Usa Python con CPU y ejecuta las celdas de arriba abajo. No necesitas GPU ni subir archivos.
3. Antes de cada cálculo, responde a la pregunta del bloque. Después explica si el resultado coincide con tu predicción.

La primera celda prepara scikit-learn; las funciones están incluidas en la siguiente.
Puedes desplegarlas para estudiar el código. Los datos vienen con la biblioteca.
Si se reinicia la sesión, vuelve a ejecutar desde el principio.

Ten abierto el **manual teórico del Tema 2 facilitado por el docente**. Las páginas
indicadas son las impresas; el visor PDF cuenta además la portada. El manual se
distribuye por separado. Las semillas y las particiones se mantienen fijas para
que puedas cotejar tus resultados con sus tablas.

In [ ]:
#@title Preparar el entorno
import importlib.metadata as metadata
import subprocess
import sys
if sys.version_info < (3,11):
    raise RuntimeError('Usa un entorno de Python 3.11 o posterior; en Colab, selecciona un entorno reciente.')
try:
    version = metadata.version('scikit-learn')
except metadata.PackageNotFoundError:
    version = None
if version != '1.9.0':
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn==1.9.0'])
print('Entorno preparado. Ejecuta ahora las funciones y continúa en orden.')

In [ ]:
#@title Funciones del ejercicio: despliega para leer las operaciones
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy import stats, linalg
from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, roc_auc_score, roc_curve
NOMBRES_IRIS = ['Long. sépalo', 'Ancho sépalo', 'Long. pétalo', 'Ancho pétalo']
COLORES = ['#0072B2', '#D55E00', '#009E73']
from IPython.display import display

def matrices_dispersion(X, y):
    """Sumas de productos sin dividir por grados de libertad."""
    media = X.mean(axis=0)
    dentro = np.zeros((X.shape[1], X.shape[1]))
    entre = np.zeros_like(dentro)
    for clase in np.unique(y):
        grupo = X[y == clase]
        mu = grupo.mean(axis=0)
        dentro += (grupo-mu).T @ (grupo-mu)
        entre += len(grupo)*np.outer(mu-media, mu-media)
    total = (X-media).T @ (X-media)
    np.testing.assert_allclose(dentro+entre, total, atol=1e-10)
    return dentro, entre, total

def caso_iris():
    datos = load_iris()
    X, y = datos.data, datos.target
    df = pd.DataFrame(X, columns=NOMBRES_IRIS)
    df['Especie'] = datos.target_names[y]
    modelo = LinearDiscriminantAnalysis(n_components=2, solver='svd').fit(X, y)
    Z = modelo.transform(X)
    Sw, Sb, St = matrices_dispersion(X, y)
    valores = linalg.eigvalsh(Sb, Sw)[::-1][:2]
    np.testing.assert_allclose(valores/valores.sum(), modelo.explained_variance_ratio_)
    np.testing.assert_allclose(Z, (X-modelo.xbar_) @ modelo.scalings_)
    np.testing.assert_allclose(modelo.scalings_.T @ (Sw/(len(y)-3)) @ modelo.scalings_, np.eye(2), atol=1e-10)
    centroides = np.array([Z[y == k].mean(axis=0) for k in range(3)])
    distancias = pd.DataFrame([
        {'Par': f'{datos.target_names[i]}–{datos.target_names[j]}',
         'Distancia': np.linalg.norm(centroides[i]-centroides[j])}
        for i,j in [(0,1),(0,2),(1,2)]])
    return dict(X=X, y=y, df=df, datos=datos, modelo=modelo, Z=Z,
                Sw=Sw, Sb=Sb, St=St, autovalores=valores,
                medias=df.groupby('Especie')[NOMBRES_IRIS].mean(),
                desviaciones=df.groupby('Especie')[NOMBRES_IRIS].std(ddof=1),
                centroides=centroides, distancias=distancias)

def relacion_anova_lda(caso):
    """Comparaciones univariantes exploratorias; CV estratificada sin barajar."""
    X, y = caso['X'], caso['y']
    filas=[]
    for j, nombre in enumerate(NOMBRES_IRIS):
        grupos = [X[y==k,j] for k in range(3)]
        F,p = stats.f_oneway(*grupos)
        sw = sum(np.sum((a-a.mean())**2) for a in grupos)
        st = np.sum((X[:,j]-X[:,j].mean())**2)
        scores = cross_val_score(LinearDiscriminantAnalysis(), X[:,[j]], y, cv=5)
        filas.append(dict(Variable=nombre,F=F,p=p,Lambda=sw/st,Exactitud_CV=scores.mean()))
    return pd.DataFrame(filas).sort_values('F',ascending=False).reset_index(drop=True)

def wilks_permutacion(X, y, B=1999, semilla=20260908):
    """Cola inferior de Lambda, conservando tamaños, bajo intercambiabilidad."""
    Sw, _, St = matrices_dispersion(X,y)
    signo, log_total = np.linalg.slogdet(St)
    assert signo > 0
    observado = np.linalg.slogdet(Sw)[1]-log_total
    rng = np.random.default_rng(semilla)
    extremos=0
    for _ in range(B):
        W,_,_ = matrices_dispersion(X,rng.permutation(y))
        extremos += np.linalg.slogdet(W)[1]-log_total <= observado+1e-12
    return dict(Lambda=float(np.exp(observado)),p_MC=(int(extremos)+1)/(B+1),
                B=B,extremos=int(extremos),semilla=semilla)

def intervalo_wilson(aciertos, n, confianza=.95):
    z = stats.norm.ppf((1+confianza)/2)
    p = aciertos/n
    centro = (p+z*z/(2*n))/(1+z*z/n)
    margen = z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/(1+z*z/n)
    return float(centro-margen), float(centro+margen)

def evaluacion_iris(caso):
    Xtr, Xte, ytr, yte = train_test_split(caso['X'],caso['y'],test_size=.3,random_state=42,stratify=caso['y'])
    modelo = LinearDiscriminantAnalysis().fit(Xtr,ytr)
    pred = modelo.predict(Xte)
    matriz = confusion_matrix(yte,pred,labels=[0,1,2])
    informe = pd.DataFrame(classification_report(yte,pred,target_names=caso['datos'].target_names,output_dict=True)).T
    aciertos = int(np.trace(matriz))
    return dict(modelo=modelo,X_test=Xte,y_test=yte,pred=pred,matriz=matriz,informe=informe,
                exactitud=accuracy_score(yte,pred),IC=intervalo_wilson(aciertos,len(yte)))

def elipse_descriptiva(ax, puntos, color, cobertura=.95):
    """Elipse gaussiana aproximada de observaciones; no IC del centroide."""
    valores, vectores = np.linalg.eigh(np.cov(puntos,rowvar=False,ddof=1))
    orden = np.argsort(valores)[::-1]
    valores, vectores = valores[orden], vectores[:,orden]
    radio = np.sqrt(stats.chi2.ppf(cobertura,df=2))
    angulo = np.degrees(np.arctan2(vectores[1,0],vectores[0,0]))
    ax.add_patch(Ellipse(puntos.mean(axis=0),width=2*radio*np.sqrt(valores[0]),
        height=2*radio*np.sqrt(valores[1]),angle=angulo,facecolor=color,edgecolor=color,alpha=.12))

def figura_geometria(caso):
    modelo,Z,y = caso['modelo'],caso['Z'],caso['y']
    porcentajes=100*modelo.explained_variance_ratio_
    fig, axes=plt.subplots(1,2,figsize=(11,4.4),layout='constrained')
    for k,color in enumerate(COLORES):
        puntos=Z[y==k]
        elipse_descriptiva(axes[0],puntos,color)
        axes[0].scatter(*puntos.T,s=19,color=color,label=caso['datos'].target_names[k],alpha=.8)
        axes[0].scatter(*puntos.mean(axis=0),marker='*',s=170,color=color,edgecolor='black')
    axes[0].set(xlabel=f'LD1 ({porcentajes[0]:.2f} %)',ylabel=f'LD2 ({porcentajes[1]:.2f} %)',
                title='Proyección del ajuste completo de Iris')
    axes[0].legend(fontsize=9)
    axes[1].bar(['LD1','LD2'],porcentajes,color=COLORES[:2])
    for j,p in enumerate(porcentajes): axes[1].text(j,p+1.5,f'{p:.2f} %',ha='center')
    axes[1].set(ylim=(0,112),ylabel='Proporción discriminante (%)',title='Autovalores normalizados')
    return fig

def figura_coeficientes(caso):
    """Coeficientes en unidades originales y correlaciones intraclase."""
    X,y,Z=caso['X'],caso['y'],caso['Z']
    Xr=np.zeros_like(X); Zr=np.zeros_like(Z)
    for k in range(3):
        Xr[y==k]=X[y==k]-X[y==k].mean(axis=0)
        Zr[y==k]=Z[y==k]-Z[y==k].mean(axis=0)
    estructura=np.corrcoef(np.column_stack([Xr,Zr]),rowvar=False)[:4,4:]
    fig,axes=plt.subplots(1,2,figsize=(11,4.3),layout='constrained')
    posiciones=np.arange(4)
    for j in range(2):
        axes[0].barh(posiciones+(j-.5)*.3,caso['modelo'].scalings_[:,j],height=.3,color=COLORES[j],label=f'LD{j+1}')
    axes[0].set(yticks=posiciones,yticklabels=NOMBRES_IRIS,xlabel='Coeficiente (variables en cm)',title='Direcciones: el signo del eje es arbitrario')
    axes[0].axvline(0,color='gray',lw=.7); axes[0].legend()
    im=axes[1].imshow(estructura,vmin=-1,vmax=1,cmap='RdBu_r',aspect='auto')
    axes[1].set(xticks=[0,1],xticklabels=['LD1','LD2'],yticks=posiciones,yticklabels=NOMBRES_IRIS,title='Correlaciones de estructura intraclase')
    for i in range(4):
        for j in range(2): axes[1].text(j,i,f'{estructura[i,j]:.2f}',ha='center',va='center',color='white' if abs(estructura[i,j])>.6 else 'black')
    fig.colorbar(im,ax=axes[1],shrink=.8)
    return fig, pd.DataFrame(estructura,index=NOMBRES_IRIS,columns=['LD1','LD2'])

def figura_supuestos(caso):
    fig,axes=plt.subplots(1,2,figsize=(10,4),layout='constrained')
    for k,color in enumerate(COLORES):
        a=caso['X'][caso['y']==k,0]
        axes[0].hist(a,bins=np.linspace(4,8,15),histtype='step',lw=2,color=color,label=caso['datos'].target_names[k])
        teoricos,observados=stats.probplot((a-a.mean())/a.std(ddof=1),fit=False)
        axes[1].scatter(teoricos,observados,s=15,color=color)
    axes[0].set(xlabel='Longitud del sépalo (cm)',ylabel='Frecuencia',title='Distribuciones dentro de cada especie')
    axes[0].legend(fontsize=9)
    axes[1].plot([-2.5,2.5],[-2.5,2.5],color='gray',ls='--')
    axes[1].set(xlabel='Cuantil normal',ylabel='Cuantil observado estandarizado',title='Q–Q univariante: diagnóstico parcial')
    return fig

def figura_confusion(matriz,nombres,titulo):
    fig,ax=plt.subplots(figsize=(5,4.2),layout='constrained')
    ax.imshow(matriz,cmap='Blues',vmin=0)
    for i in range(len(nombres)):
        for j in range(len(nombres)):
            ax.text(j,i,str(matriz[i,j]),ha='center',va='center',fontsize=14,color='white' if matriz[i,j]>matriz.max()/2 else 'black')
    ax.set(xticks=range(len(nombres)),xticklabels=nombres,yticks=range(len(nombres)),yticklabels=nombres,
           xlabel='Clase predicha',ylabel='Clase real',title=titulo)
    return fig

pd.set_option("display.precision", 5)
plt.rcParams.update({"font.size":10,"axes.spines.top":False,"axes.spines.right":False})

## 1. Conoce a las 150 flores

📖 **Manual del Tema 2:** apartado 6.1, p. 11 (material facilitado por el docente).

Cada fila es una flor y las cuatro medidas están en centímetros. Calcularemos las medias y desviaciones por especie: una fila individual no puede sustituir a la media de sus 50 flores. Este ajuste completo se utilizará para explicar la geometría; más adelante entrenaremos otro modelo para evaluar en test.

💭 **Antes de ejecutar:** Si en la primera fila aparece 5,1 cm, ¿puedes decir que esa es la media de setosa?

In [ ]:
iris = caso_iris()
display(iris['df'].head())
display(iris['df'].groupby('Especie').size().rename('Número de flores'))
display(pd.concat({'Media':iris['medias'], 'DE muestral':iris['desviaciones']}, axis=1))

## 2. Una conexión con F del Tema 1

📖 **Manual del Tema 2:** apartado 6.2, p. 11 (material facilitado por el docente).

Probamos cada variable por separado. F y Lambda describen diferencias de grupos; la exactitud CV evalúa predicciones. Aquí usamos cinco pliegues estratificados sin barajar para reproducir el ejercicio. Es una exploración; seleccionar variables con estas salidas y evaluar después con las mismas etiquetas no ofrece una validación independiente.

💭 **Antes de ejecutar:** ¿Una variable poco útil por separado podría aportar información al combinarse con otra?

In [ ]:
tabla_anova = relacion_anova_lda(iris)
display(tabla_anova)
ancho = tabla_anova.loc[tabla_anova.Variable=='Ancho sépalo','Exactitud_CV'].iloc[0]
print(f'Exactitud CV usando solo el ancho del sépalo: {100*ancho:.2f} %.')
print('Exactitud es la fracción total de aciertos; precisión es TP/(TP+FP).')

## 3. Examina los supuestos dentro de cada grupo

📖 **Manual del Tema 2:** apartado 6.3, p. 12 (material facilitado por el docente).

Observa la distribución y el Q–Q por especie. Después compara covarianzas. Una gráfica univariante no demuestra normalidad multivariante y tres matrices muestrales diferentes no bastan para concluir que las poblacionales difieren. La independencia depende del diseño.

💭 **Antes de ejecutar:** ¿Por qué un histograma con todas las especies mezcladas no comprueba la normalidad dentro de clase?

In [ ]:
figura_supuestos(iris)
plt.show()
covarianzas = {nombre:pd.DataFrame(np.cov(iris['X'][iris['y']==k],rowvar=False),
                                  index=NOMBRES_IRIS,columns=NOMBRES_IRIS)
               for k,nombre in enumerate(iris['datos'].target_names)}
display(pd.concat(covarianzas,names=['Especie','Variable']))
print(f"Número de condición de S_W: {np.linalg.cond(iris['Sw']):.2f}")

## 4. Construye las matrices y encuentra las direcciones

📖 **Manual del Tema 2:** apartado 7.1, p. 13 (material facilitado por el docente).

S_W y S_B son sumas de productos: todavía no se han dividido por grados de libertad. Su suma debe coincidir con S_T. Resolvemos S_B w = λ S_W w, sin formar una inversa explícita. Con tres clases y cuatro variables hay como máximo dos direcciones discriminantes.

💭 **Antes de ejecutar:** ¿Cuántos ejes podrías obtener si conservaras solo dos especies?

In [ ]:
for etiqueta in ['Sw','Sb','St']:
    print(etiqueta)
    display(pd.DataFrame(iris[etiqueta],index=NOMBRES_IRIS,columns=NOMBRES_IRIS))
display(pd.DataFrame({'Autovalor':iris['autovalores'],
                      'Proporción':iris['modelo'].explained_variance_ratio_},index=['LD1','LD2']))
np.testing.assert_allclose(iris['Sw']+iris['Sb'],iris['St'])

## 5. Proyecta con la misma escala del ajuste

📖 **Manual del Tema 2:** apartado 7.3, p. 13 (material facilitado por el docente).

Usamos el modelo ajustado sobre las variables originales. Con el solucionador SVD, transform centra con xbar_ y aplica scalings_. Si estandarizases las variables, tendrías que ajustar también el modelo en esa escala.

💭 **Antes de ejecutar:** ¿Por qué no debemos multiplicar datos estandarizados por coeficientes obtenidos en centímetros?

In [ ]:
lda = iris['modelo']
W = lda.scalings_
Z = lda.transform(iris['X'])
np.testing.assert_allclose(Z,(iris['X']-lda.xbar_) @ W)
display(pd.DataFrame(W,index=NOMBRES_IRIS,columns=['LD1','LD2']))
display(pd.DataFrame(Z[:5],columns=['LD1','LD2']))
print('Primera flor:',iris['X'][0], '→ coordenadas:',Z[0])
normalizacion = W.T @ (iris['Sw']/(len(iris['y'])-3)) @ W
display(pd.DataFrame(normalizacion,index=['LD1','LD2'],columns=['LD1','LD2']))
print('La identidad utiliza la covarianza intraclase S_W/(N−g). Con S_W se obtiene 147 I.')

## 6. Lee los porcentajes, los centroides y las elipses

📖 **Manual del Tema 2:** apartado 8.1, p. 16 (material facilitado por el docente).

Los porcentajes son autovalores normalizados; no se calculan sumando magnitudes de coeficientes. Las elipses usan radio √χ²(2;0,95) y describen dispersión gaussiana aproximada de observaciones. No son intervalos de confianza del centroide. Consulta también 8.2 y 8.3.

💭 **Antes de ejecutar:** Si LD2 explica una proporción pequeña del criterio, ¿significa que debas borrarlo sin comprobar su utilidad?

In [ ]:
figura_geometria(iris)
plt.show()
display(iris['distancias'])
print('Proporciones discriminantes (%):',100*lda.explained_variance_ratio_)
print(f'Radio para elipse gaussiana del 95 %: {np.sqrt(stats.chi2.ppf(.95,2)):.4f}')
print(f'Cobertura poblacional normal bivariante con radio 2: {100*stats.chi2.cdf(4,2):.2f} %')

## 7. Coeficientes y correlaciones cuentan cosas distintas

📖 **Manual del Tema 2:** apartado 7.6, p. 15 (material facilitado por el docente).

La figura de la izquierda conserva las unidades originales. A la derecha calculamos correlaciones de estructura después de centrar dentro de cada especie. Invertir el signo de un eje no cambia la solución. Un coeficiente grande no demuestra importancia causal.

💭 **Antes de ejecutar:** ¿Se pueden comparar sin más las magnitudes de coeficientes de variables medidas en unidades distintas?

In [ ]:
fig, estructura = figura_coeficientes(iris)
plt.show()
display(estructura)

## 8. Pasar de una flor a una clase

📖 **Manual del Tema 2:** apartado 7.5, p. 14 (material facilitado por el docente).

Para Iris hay tres puntuaciones de clasificación y dos coordenadas discriminantes. Mostramos coef_, las puntuaciones y las probabilidades para una flor del ajuste descriptivo. La decisión usa la clase de máxima puntuación; la calidad predictiva se evaluará en el bloque siguiente.

💭 **Antes de ejecutar:** ¿Esperas que coef_ tenga la misma forma que scalings_? Explica por qué.

In [ ]:
display(pd.DataFrame(lda.coef_,index=iris['datos'].target_names,columns=NOMBRES_IRIS))
flor = iris['X'][[0]]
puntuaciones = flor @ lda.coef_.T + lda.intercept_
np.testing.assert_allclose(puntuaciones,lda.decision_function(flor))
display(pd.DataFrame({'Puntuación':puntuaciones[0],
                      'Probabilidad':lda.predict_proba(flor)[0]},index=iris['datos'].target_names))
print('Clase predicha:',iris['datos'].target_names[lda.predict(flor)[0]])

## 9. Evaluar con flores que el modelo no ha visto

📖 **Manual del Tema 2:** apartado 9.1, p. 18 (material facilitado por el docente).

Ahora entrenamos un modelo nuevo con 105 flores y reservamos 45. Matriz e informe salen de las mismas predicciones. El IC de Wilson describe incertidumbre de la proporción bajo sus supuestos, condicionado al ajuste; no incluye toda la variación de un nuevo entrenamiento.

💭 **Antes de ejecutar:** ¿Por qué la figura ajustada sobre las 150 flores no sustituye a esta evaluación?

In [ ]:
evaluacion = evaluacion_iris(iris)
figura_confusion(evaluacion['matriz'],list(iris['datos'].target_names),'Test independiente del ajuste: 45 flores')
plt.show()
display(evaluacion['informe'].loc[iris['datos'].target_names])
print(f"Exactitud: {100*evaluacion['exactitud']:.2f} %; IC Wilson 95 %: "
      f"[{100*evaluacion['IC'][0]:.2f}, {100*evaluacion['IC'][1]:.2f}] %")

## 10. Wilks: una pregunta inferencial diferente

📖 **Manual del Tema 2:** apartado 9.3, p. 19 (material facilitado por el docente).

Usamos det(S_W)/det(S_T). Para ilustrar un contraste, permutamos etiquetas 1999 veces conservando tamaños y contamos la cola de Lambda pequeña. La permutación requiere intercambiabilidad; bajo el modelo gaussiano de covarianza común, H₀ de medias iguales la implica. Esta prueba no evalúa predicción.

💭 **Antes de ejecutar:** Si no aparece ninguna permutación más extrema, ¿puedes decir que el p-valor es cero?

In [ ]:
wilks = wilks_permutacion(iris['X'],iris['y'])
display(pd.Series(wilks,name='Contraste global por permutación'))
np.testing.assert_allclose(wilks['Lambda'],np.prod(1/(1+iris['autovalores'])))
print('Se usa (b+1)/(B+1). El mínimo resoluble no es el p-valor exacto.')

## Tu conclusión, en cinco líneas

Indica qué se ha proyectado, qué separa LD1, qué informa Wilks y qué exactitud
obtiene el modelo en test. Añade una limitación y distingue exactitud de precisión.
Si utilizas IA para revisar tu texto, pídele que compruebe tus cifras con las salidas
y que no confunda una proyección descriptiva con validación. Guarda la copia
ejecutada y descarga el `.ipynb` desde Archivo si debes entregarlo.